In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ==========================================
# 1. LOAD DATASET KOMDIGI
# ==========================================
data_candidates = [
    Path("data/komdigi_hoaks.csv"),
    Path("../../../data/komdigi_hoaks.csv"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("File data/komdigi_hoaks.csv tidak ditemukan.")

dataset = pd.read_csv(DATA_PATH, usecols=["title", "body_text", "topics"])
dataset["text"] = dataset["title"].fillna("") + " " + dataset["body_text"].fillna("")
dataset["topics"] = dataset["topics"].fillna("").str.strip()
dataset = dataset.loc[dataset["text"].str.strip().ne("") & dataset["topics"].ne("")].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(dataset["topics"])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    dataset["text"],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF hanya di-fit pada data latih untuk mencegah kebocoran data.
vectorizer = TfidfVectorizer(max_features=512, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)
num_classes = len(label_encoder.classes_)
class_counts = np.bincount(y_train, minlength=num_classes)
class_weights = {
    class_id: len(y_train) / (num_classes * count)
    for class_id, count in enumerate(class_counts)
}

print(f"Jumlah data: {len(dataset)}")
print(f"Jumlah fitur TF-IDF: {X_train.shape[1]}")
print("Kelas topik:", list(label_encoder.classes_))

# ==========================================
# 2. FUNGSI MEMBANGUN DAN MELATIH MODEL
# ==========================================
def train_model(name, neurons, activation):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(neurons[0], activation=activation),
        tf.keras.layers.Dense(neurons[1], activation=activation),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(f"\n=== {name} ===")
    print(f"Neuron hidden layer: {neurons}")
    print(f"Aktivasi: {activation}")

    model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=32,
        class_weight=class_weights,
        verbose=0
    )

    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    predictions = np.argmax(model.predict(X_test, verbose=0), axis=1)
    macro_f1 = f1_score(y_test, predictions, average="macro", zero_division=0)

    print(f"Test Loss     : {loss:.4f}")
    print(f"Test Accuracy : {accuracy:.4f}")
    print(f"Test Macro-F1 : {macro_f1:.4f}")

    return {
        "Percobaan": name,
        "Neuron Hidden Layer": str(neurons),
        "Aktivasi": activation,
        "Loss": loss,
        "Akurasi": accuracy,
        "Macro-F1": macro_f1,
        "Prediksi": predictions
    }


# ==========================================
# 3. PERBANDINGAN JUMLAH NEURON
# ==========================================
results = []
results.append(train_model("Konfigurasi Awal", [10, 8], "relu"))
results.append(train_model("Hidden Layer Lebih Besar", [16, 12], "relu"))

# ==========================================
# 4. PERBANDINGAN SIGMOID VS RELU
# ==========================================
results.append(train_model("Aktivasi Sigmoid", [10, 8], "sigmoid"))
results.append(train_model("Aktivasi ReLU", [10, 8], "relu"))

# ==========================================
# 5. PERBANDINGAN DAN LAPORAN HASIL
# ==========================================
results_df = pd.DataFrame([
    {key: value for key, value in result.items() if key != "Prediksi"}
    for result in results
])
print("\n=== TABEL PERBANDINGAN HASIL ===")
print(results_df.to_string(index=False))

best_result = max(results, key=lambda result: result["Macro-F1"])
print(f"\n=== Laporan kelas: {best_result['Percobaan']} ===")
print(classification_report(
    y_test,
    best_result["Prediksi"],
    labels=np.arange(num_classes),
    target_names=label_encoder.classes_,
    zero_division=0
))
